Python, Spark and JVM version

In [1]:
import sys
import pyspark
import py4j

print(sys.executable)
print("PySpark:", pyspark.__version__)
print("Py4J:", py4j.__version__)

/usr/bin/python3
PySpark: 4.2.0
Py4J: 0.10.9.9


Create Spark Session

In [ ]:
import pyspark

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (
    SparkSession.builder
    .appName("DeltaLab")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /tmp/spark-home/.ivy2.5.2/cache
The jars for the packages stored in: /tmp/spark-home/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3c4e618d-3203-4c21-972b-1679a993ec0e;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.0 in central
	found org.roaringbitm

Turning on the AQE (Adaptive Query Execution)

In [ ]:

#Uncomment the code the turn it on
# spark.conf.set("spark.sql.adaptive.enabled", "true")

Sample spark code 

In [4]:
from pyspark.sql import functions as F

# --------------------------------------------------
# 1. Create a reasonably large distributed dataset
# --------------------------------------------------

df = (
    spark.range(0, 5_000_000, 1, numPartitions=8)
    .withColumn("group_id", (F.col("id") % 1000).cast("int"))
    .withColumn("value", (F.col("id") * 10) % 5000)
)

print("Initial partitions:", df.rdd.getNumPartitions())


# --------------------------------------------------
# 2. Force a shuffle using repartition
# --------------------------------------------------

repartitioned = df.repartition(16, "group_id")

print("After repartition:", repartitioned.rdd.getNumPartitions())


# --------------------------------------------------
# 3. Create a second dataset for a join
# --------------------------------------------------

lookup = (
    spark.range(0, 1000)
    .withColumnRenamed("id", "group_id")
    .withColumn(
        "category",
        F.concat(F.lit("category_"), F.col("group_id"))
    )
)


# --------------------------------------------------
# 4. Disable broadcast so Spark performs a shuffle join
# --------------------------------------------------

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

joined = repartitioned.join(
    lookup,
    on="group_id",
    how="inner"
)


# --------------------------------------------------
# 5. Aggregate
# --------------------------------------------------

aggregated = (
    joined
    .groupBy("category")
    .agg(
        F.count("*").alias("row_count"),
        F.sum("value").alias("total_value"),
        F.avg("value").alias("avg_value")
    )
)


# --------------------------------------------------
# 6. Sort
# --------------------------------------------------

result = aggregated.orderBy(
    F.desc("total_value")
)


# --------------------------------------------------
# 7. Write result as Delta
# --------------------------------------------------

output_path = "/workspace/data/delta/category_summary"

spark.sparkContext.setJobGroup(
    "delta-category-summary",
    "Join, aggregate, sort, and write category summary as Delta"
)

result.write \
    .format("delta") \
    .mode("overwrite") \
    .save(output_path)

print(f"Delta table written to: {output_path}")

Initial partitions: 8


[Stage 1:============================================>              (6 + 2) / 8]

After repartition: 16


Delta table written to: /workspace/data/delta/category_summary


Read the Delta tables 

delta_df = (
    spark.read
    .format("delta")
    .load("/data/delta/category_summary")
)

delta_df.show(50, truncate=False)